In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DATA_DIR            = Path("../data")
MODEL_PATH          = DATA_DIR / "model.pkl"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "mlruns")
EXPERIMENT_NAME     = "csgo-match-predictor"
MODEL_NAME          = "csgo-match-predictor"
ACCURACY_THRESHOLD  = 0.60

In [ ]:
import joblib
import json

model = joblib.load(MODEL_PATH)

with open(DATA_DIR / "evaluation.json") as f:
    metrics = json.load(f)

print(f"Model   : {type(model).__name__}")
print(f"Metrics : {metrics}")
print(f"Passes threshold ({ACCURACY_THRESHOLD}): {metrics['accuracy'] >= ACCURACY_THRESHOLD}")

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="upload") as run:
    mlflow.log_metrics(metrics)
    mlflow.log_params({"model_type": type(model).__name__})
    mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )

print(f"Registered '{MODEL_NAME}' (run {run.info.run_id})")

In [ ]:
from mlflow.tracking import MlflowClient

client   = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest   = versions[0]

if metrics["accuracy"] >= ACCURACY_THRESHOLD:
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest.version,
        stage="Staging",
    )
    print(f"Version {latest.version} → Staging")
else:
    print(f"Accuracy {metrics['accuracy']:.4f} below {ACCURACY_THRESHOLD} — not promoted")